# Vector DB — Part 1: Embed & Store

This notebook builds the TechVault knowledge base:
1. Load the four TechVault documents
2. Split each into paragraph chunks
3. Encode every chunk into a 384-dimensional vector with `all-MiniLM-L6-v2`
4. Store chunks + embeddings in ChromaDB

Run `vectordb-query.ipynb` next to search the collection.

In [1]:
%pip install -q chromadb sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

/Users/ashishb/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/ashishb/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the embedding model

`all-MiniLM-L6-v2` is fast, free, and outputs 384-dimensional vectors.

**Critical rule:** every query must use this exact same model. Use a different model at query time and the vectors live in completely different spaces — your search breaks.

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding dimensions: {model.get_sentence_embedding_dimension()}")

Embedding dimensions: 384


## 2. Create the ChromaDB collection

Using `PersistentClient` so the collection survives between notebooks.
The data is written to `./chroma_db/` on disk.

In [4]:
client = chromadb.PersistentClient(path="./chroma_db")

# Delete existing collection if re-running
try:
    client.delete_collection("techvault-docs")
except Exception:
    pass

collection = client.create_collection("techvault-docs")
print("Collection created: techvault-docs")

Collection created: techvault-docs


## 3. Load and chunk the TechVault documents

Four documents:
- Employee Handbook 2025
- Expense Policy 2025
- Product Specs: CloudDrive
- Q1 2025 All-Hands Notes

Chunking strategy: split on double newlines (one chunk per paragraph). Chunking strategy is a topic of its own — a later video in this series goes much deeper. For now, paragraphs work fine.

In [5]:
docs_dir = "../data"

documents = []
ids = []

for i, filename in enumerate(sorted(os.listdir(docs_dir))):
    if filename.endswith(".txt"):
        filepath = os.path.join(docs_dir, filename)
        with open(filepath) as f:
            text = f.read()
        # Split into chunks by paragraph
        chunks = [c.strip() for c in text.split("\n\n") if c.strip()]
        for j, chunk in enumerate(chunks):
            documents.append(chunk)
            ids.append(f"doc{i}-chunk{j}")
        print(f"{filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(documents)}")


Employee_Handbook_2025.txt: 51 chunks
Expense_Policy_2025.txt: 68 chunks
Product_Specs_CloudDrive.txt: 56 chunks
Q1_2025_AllHands_Notes.txt: 72 chunks

Total chunks: 247


## 4. Encode every chunk into embeddings

Each chunk becomes a list of 384 numbers. The four files become ~247 chunks.

In [6]:
embeddings = model.encode(documents, show_progress_bar=True).tolist()

print(f"\nEmbedding shape: {len(embeddings)} vectors × {len(embeddings[0])} dimensions")

Batches: 100%|██████████| 8/8 [00:01<00:00,  4.70it/s]


Embedding shape: 247 vectors × 384 dimensions


## 5. Store in ChromaDB

`collection.add` stores each chunk, its embedding, and its ID. ChromaDB builds the search index automatically.

In [7]:
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids,
)

print(f"Stored {collection.count()} chunks")
# Output: Stored 247 chunks

Stored 247 chunks


## 6. Spot-check a few stored chunks

In [8]:
sample = collection.get(limit=3)
for doc_id, doc in zip(sample["ids"], sample["documents"]):
    print(f"[{doc_id}] {doc[:120]}...")
    print()

[doc0-chunk0] TECHVAULT INC.
EMPLOYEE HANDBOOK 2025
Effective Date: January 1, 2025...

[doc0-chunk1] ================================================================================
TABLE OF CONTENTS
=====================...

[doc0-chunk2] ================================================================================
1. WELCOME TO TECHVAULT
===============...



---

The knowledge base is in. **247 chunks stored, embeddings indexed.**

Open `vectordb-query.ipynb` to run semantic search against this collection.